# Importing and manipulating the truss dataset

This notebook shows the lightweight base dataset import workflow, processed-cache reuse, geometry filtering with `pre_filter`, and result scaling with both `pre_transform` and runtime `transform`.

In [ ]:
from pathlib import Path
from functools import partial
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
elif not (PROJECT_ROOT / "src").exists():
    raise RuntimeError("Run this notebook from root or root/notebooks.")

SRC_ROOT = PROJECT_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from dataset import TrussBaseDataset

DATA_ROOT = PROJECT_ROOT / "data"
print("Project root:", PROJECT_ROOT)

## Selection options

The constructor below selects by typology, panel count, exact seed, and maximum designs per seed. If a selected typology folder is missing under `data/raw`, the dataset uses PyG's `download()` hook to download and unzip it from Zenodo.

In [ ]:
TYPOLOGIES = ["Fink"]      # Example: ["Pratt", "Howe"]
PANELS = [8]                # Example: [6, 8, 10]
SEEDS = [7]                 # Exact seed folder selection; set to None for all seeds.
MAX_SEEDS_PER_PANEL = None  # Optional deterministic seed cap after exact seed filtering.
MAX_DESIGNS_PER_SEED = 25   # Keep this small for a quick tutorial run.
SAMPLING_SEED = 42
PLANE = "xz"

BASE_CACHE_NAME = "tutorial_base_import"

## First construction: download and process if needed

Run this cell once. If `data/raw/Fink` is not present, the raw zip is downloaded and unzipped first. Then the selected JSON files are converted to PyG `Data` objects and cached under `data/processed`.

In [ ]:
dataset = TrussBaseDataset(
    root=DATA_ROOT,
    typologies=TYPOLOGIES,
    panels=PANELS,
    seeds=SEEDS,
    max_seeds_per_panel=MAX_SEEDS_PER_PANEL,
    max_designs_per_seed=MAX_DESIGNS_PER_SEED,
    sampling_seed=SAMPLING_SEED,
    plane=PLANE,
    cache_name=BASE_CACHE_NAME,
    force_reload=False,
)

print(f"Graphs: {len(dataset):,}")
print("Processed cache:", dataset.processed_paths[0])
print("Selection:", dataset.metadata["config"])
print("Node features:", dataset.metadata["node_feature_names"])
print("Member features:", dataset.metadata["member_feature_names"])

## Second construction: reuse the processed cache

Run this cell after the previous one. Because the configuration and `cache_name` are identical, PyG loads the processed file directly instead of parsing JSON again.

In [ ]:
cached_dataset = TrussBaseDataset(
    root=DATA_ROOT,
    typologies=TYPOLOGIES,
    panels=PANELS,
    seeds=SEEDS,
    max_seeds_per_panel=MAX_SEEDS_PER_PANEL,
    max_designs_per_seed=MAX_DESIGNS_PER_SEED,
    sampling_seed=SAMPLING_SEED,
    plane=PLANE,
    cache_name=BASE_CACHE_NAME,
    force_reload=False,
)

print(f"Graphs loaded from cache: {len(cached_dataset):,}")
print("Same processed cache:", cached_dataset.processed_paths[0])

## Inspect one graph

`TrussBaseDataset` stores node features in `x`, physical member features in `member_attr`, and metadata needed for grouping and identification.

In [ ]:
graph = cached_dataset[0]
print(graph)

summary_rows = []
for item in cached_dataset:
    summary_rows.append({
        "graph_id": int(item.graph_id.item()),
        "panels": int(item.panels.item()),
        "seed": int(item.seed.item()),
        "nodes": item.num_nodes,
        "members": int(item.num_members.item()),
    })
summary = pd.DataFrame(summary_rows)
display(summary.head())

## `pre_filter`: keep graphs above a minimum angle

`pre_filter` runs during processing, before collation and caching. Use a distinct `cache_name` or `force_reload=True` when changing a preprocessing criterion.

In [ ]:
def graph_coordinates(data: Data) -> torch.Tensor:
    """Return in-plane node coordinates from either base or task datasets."""
    if hasattr(data, "pos_raw"):
        return data.pos_raw
    return data.x[:, BASE_COORD_SLICE]
    
def minimum_member_angle_degrees(data: Data) -> float:
    """Return the smallest angle between incident members at any node."""
    coordinates = graph_coordinates(data)
    member_index = data.member_index
    vectors_by_node: list[list[torch.Tensor]] = [[] for _ in range(data.num_nodes)]

    for start, end in member_index.t().tolist():
        vector = coordinates[end] - coordinates[start]
        vectors_by_node[start].append(vector)
        vectors_by_node[end].append(-vector)

    minimum = 180.0
    for vectors in vectors_by_node:
        if len(vectors) < 2:
            continue
        for i, first in enumerate(vectors[:-1]):
            first_norm = first.norm().clamp_min(1e-12)
            for second in vectors[i + 1 :]:
                cosine = torch.dot(first, second) / (
                    first_norm * second.norm().clamp_min(1e-12)
                )
                angle = math.degrees(math.acos(float(cosine.clamp(-1.0, 1.0))))
                minimum = min(minimum, angle)
    return minimum


def minimum_angle_filter(minimum_degrees: float):
    """Return a PyG ``pre_filter`` callable for minimum nodal member angle."""

    def pre_filter(data: Data) -> bool:
        return minimum_member_angle_degrees(data) >= float(minimum_degrees)

    return pre_filter

In [ ]:
MIN_ANGLE_DEGREES = 20.0

angle_filtered_dataset = TrussBaseDataset(
    root=DATA_ROOT,
    typologies=TYPOLOGIES,
    panels=PANELS,
    seeds=SEEDS,
    max_seeds_per_panel=MAX_SEEDS_PER_PANEL,
    max_designs_per_seed=MAX_DESIGNS_PER_SEED,
    sampling_seed=SAMPLING_SEED,
    plane=PLANE,
    cache_name=f"tutorial_base_min_angle_{int(MIN_ANGLE_DEGREES)}",
    pre_filter=minimum_angle_filter(MIN_ANGLE_DEGREES), ####### the defined pre_filtering behavior is used here
    force_reload=False,
)

print(f"Before angle filter: {len(cached_dataset):,}")
print(f"After angle filter:  {len(angle_filtered_dataset):,}")
print("Processed cache:", angle_filtered_dataset.processed_paths[0])

## Material scaling as `pre_transform`

`pre_transform` runs once during processing and the scaled graphs are what gets saved in `data/processed`. This is the right option when the material/load case is part of the dataset definition.

In [ ]:
def scale_results(
    data: Data,
    *,
    force_scale: float,
    displacement_scale: float | None = None,
    length_scale: float = 1.0,
) -> Data:
    """Return a scaled copy of a truss graph. """
    out = data.clone()

    for name in ("pos_raw", "member_length"):
        if hasattr(out, name):
            setattr(out, name, getattr(out, name) * length_scale)
    for name in ("axial_force", "load_scale", "static_action"):
        if hasattr(out, name):
            scale = abs(force_scale) * length_scale if name == "static_action" else force_scale
            setattr(out, name, getattr(out, name) * scale)
    for name in ("displacement", "displacement_scale"):
        if hasattr(out, name):
            setattr(out, name, getattr(out, name) * displacement_scale)
    return out

class MaterialScales:
    """Scale factors for geometry and linear-elastic response quantities."""

    length: float
    force: float
    displacement: float


def material_scales(
    *,
    span: float,
    load_per_length: float,
    young_modulus: float,
    area: float,
) -> MaterialScales:
    """Return result scale factors for a uniformly loaded linear truss family."""
    total_load = float(load_per_length) * float(span)
    axial_stiffness = float(young_modulus) * float(area)
    if axial_stiffness <= 0:
        raise ValueError("young_modulus * area must be positive")
    return MaterialScales(
        length=float(span),
        force=total_load,
        displacement=total_load * float(span) / axial_stiffness,
    )

def scale_to_material(
    data: Data,
    *,
    span: float,
    load_per_length: float,
    young_modulus: float,
    area: float,
) -> Data:
    """Scale geometry, forces, and displacement to a material/load case."""
    scales = material_scales(
        span=span,
        load_per_length=load_per_length,
        young_modulus=young_modulus,
        area=area,
    )
    return scale_results(
        data,
        length_scale=scales.length,
        force_scale=scales.force,
        displacement_scale=scales.displacement,
    )


In [ ]:
SPAN = 20.0                 # m
LOAD_PER_LENGTH = 3_000.0   # N/m
YOUNG_MODULUS = 2.1e11      # N/m2
AREA = 0.004525             # m2

scales = material_scales(
    span=SPAN,
    load_per_length=LOAD_PER_LENGTH,
    young_modulus=YOUNG_MODULUS,
    area=AREA,
)
material_cache_name = (
    f"tutorial_base_material_pretransform_span_{SPAN:g}_"
    f"load_{LOAD_PER_LENGTH:g}_E_{YOUNG_MODULUS:g}_A_{AREA:g}"
).replace(".", "p")
print(scales)
print("Material cache name:", material_cache_name)

pretransformed_dataset = TrussBaseDataset(
    root=DATA_ROOT,
    typologies=TYPOLOGIES,
    panels=PANELS,
    seeds=SEEDS,
    max_seeds_per_panel=MAX_SEEDS_PER_PANEL,
    max_designs_per_seed=MAX_DESIGNS_PER_SEED,
    sampling_seed=SAMPLING_SEED,
    plane=PLANE,
    cache_name=material_cache_name,
    pre_transform=partial(       ####### the defined pre_filtering behavior is used here
        scale_to_material,
        span=SPAN,
        load_per_length=LOAD_PER_LENGTH,
        young_modulus=YOUNG_MODULUS,
        area=AREA,
    ),
    force_reload=False,
)

original = cached_dataset[0]
scaled = pretransformed_dataset[0]
print("Original first member [length, axial_force]:", original.member_attr[0].tolist())
print("Scaled first member   [length, axial_force]:", scaled.member_attr[0].tolist())
print("Original first node force/displacement:", original.x[0, 4:8].tolist())
print("Scaled first node force/displacement:  ", scaled.x[0, 4:8].tolist())

## Same scaling as runtime `transform`

`transform` runs every time a graph is accessed. The processed cache remains unscaled, which is useful when you want to reuse the same cached import for several material or visualization scenarios.

In [ ]:
runtime_scaled_dataset = TrussBaseDataset(
    root=DATA_ROOT,
    typologies=TYPOLOGIES,
    panels=PANELS,
    seeds=SEEDS,
    max_seeds_per_panel=MAX_SEEDS_PER_PANEL,
    max_designs_per_seed=MAX_DESIGNS_PER_SEED,
    sampling_seed=SAMPLING_SEED,
    plane=PLANE,
    cache_name=BASE_CACHE_NAME,
    transform=partial(
        scale_to_material,
        span=SPAN,
        load_per_length=LOAD_PER_LENGTH,
        young_modulus=YOUNG_MODULUS,
        area=AREA,
    ),
    force_reload=False,
)

runtime_scaled = runtime_scaled_dataset[0]
print("Runtime-scaled first member:", runtime_scaled.member_attr[0].tolist())
print("Cached raw first member remains:", cached_dataset[0].member_attr[0].tolist())